# FAR-Trans External Validation

## tl;dr

This notebook audits three submission-critical extensions to the primary analysis:

1. market-index definition sensitivity,
2. high-confidence corporate-action and conservative mechanical-event screens, and
3. the stability of out-of-time predictive comparisons.

The main conclusion is deliberately conservative: **revealed transaction history is consistently more useful than the stated MiFID profile; the additional value of prior extreme-event response history is sensitive to the event screen.**

## Context & Methods

### Key assumptions

- The official ATHEX benchmark and historical-source identity are recorded in `data/external/market_index_source_manifest.csv`, but the binary workbook was not downloadable in this isolated runtime.
- Four internal market proxies were therefore compared: cross-sectional median, equal-weight mean, 10% trimmed mean, and 2% winsorized mean.
- High-confidence official ATHEX corporate actions directly overlapping detected events were screened. A wider conservative screen also removes non-broad drops of at least 20%, round -20%/-30% moves, and stale-price-gap candidates.
- Model metrics are independent LightGBM robustness estimates, not replacements for the primary CatBoost results.

In [1]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path('..').resolve()
EXT = ROOT / 'results_external'
DATA_EXT = ROOT / 'data' / 'external'

sample = pd.read_csv(EXT / 'external_validation_sample_summary.csv')
metrics = pd.read_csv(EXT / 'external_validation_model_metrics.csv')
bootstrap = pd.read_csv(EXT / 'external_validation_bootstrap.csv')
overlap = pd.read_csv(EXT / 'market_proxy_event_overlap.csv')
actions = pd.read_csv(EXT / 'confirmed_action_matched_events.csv')
manifest = pd.read_csv(DATA_EXT / 'market_index_source_manifest.csv')

print('Loaded external-validation artifacts.')

Loaded external-validation artifacts.


## Data

The table below shows how the analytic population changes under each screen. The original sample is preserved for audit, while the stricter screens remove events rather than changing customer labels after the fact.

In [2]:
sample = sample.assign(action_rate=sample['actions'] / sample['exposures'])
sample[['scenario','exposures','customers','events','actions','action_rate']].style.format({'action_rate':'{:.2%}'})

,scenario,exposures,customers,events,actions,action_rate
0,original,35424,5492,552,2482,7.01%
1,official_action_screened,34433,5489,548,2409,7.00%
2,proxy_consensus_3of4,27075,5218,457,1942,7.17%
3,mechanical_conservative,31701,5383,517,1947,6.14%


### Market-proxy agreement

The median proxy reproduces the original event set by construction. The other proxies provide an independent sensitivity check. A three-of-four consensus retains only events detected by at least three definitions.

In [3]:
overlap.style.format({'original_recall':'{:.1%}','jaccard':'{:.3f}'})

,proxy,n_events,overlap_original,original_recall,jaccard
0,equal_weight_mean,607,461,83.5%,0.660
1,median,611,552,100.0%,0.903
2,trimmed_mean_10pct,603,474,85.9%,0.696
3,winsorized_mean_2pct,599,458,83.0%,0.661


### High-confidence official corporate-action matches

These are the official-event dates that directly overlap the analytic event set. The screen is intentionally narrow and does **not** claim complete coverage of every cash dividend or issuer action.

In [4]:
actions

,event_id,ISIN,event_date,raw_return,matched_action_date,issuer,action,source,source_url
0,GRS001003037|2021-04-29,GRS001003037,2021-04-29,-0.298951,2021-04-29,ATTICA BANK S.A.,Trading resumption after suspension / adjusted...,ATHEX Securities Market Information Bulletin a...,https://www.athexgroup.gr/en/market-data/issue...
1,GRS001003037|2021-09-30,GRS001003037,2021-09-30,-0.268617,2021-09-30,ATTICA BANK S.A.,Reverse share split and adjusted start price,ATHEX Securities Market Information Bulletin 2...,https://www.athexgroup.gr/en/more-options/anno...
2,GRS001003037|2021-11-22,GRS001003037,2021-11-22,-0.549367,2021-11-22,ATTICA BANK S.A.,Ex-rights trading / share capital increase pri...,ATHEX clarification of ATTICA BANK share-price...,https://www.athexgroup.gr/en/node/704182
3,GRS014003032|2021-04-20,GRS014003032,2021-04-20,-0.509615,2021-04-19,PIRAEUS FINANCIAL HOLDINGS S.A.,Capital restructuring / share capital increase...,ATHEX Piraeus issuer corporate-actions archive,https://www.athexgroup.gr/en/market-data/issue...


## Results

### Revealed behavior remains stronger than the stated profile

For every external-validation scenario, the model containing general revealed behavior (`M3`) outperforms the profile-only model (`M1`). The event-specific history bundle (`M4`) is strongest in the original and conservative mechanical screens, but is nearly tied with `M3` after the four confirmed official events are removed.

In [5]:
metric_view = metrics.pivot(index='scenario', columns='model_short', values='PR_AUC')
metric_view[['M0_event_position','M1_plus_profile','M3_profile_behavior','M4_plus_prior_shocks']].round(4)

model_short,M0_event_position,M1_plus_profile,M3_profile_behavior,M4_plus_prior_shocks
scenario,,,,
mechanical_conservative,0.2298,0.2217,0.2580,0.2746
official_action_screened,0.2215,0.2235,0.2745,0.2735
original,0.2215,0.2242,0.2563,0.2735
proxy_consensus_3of4,0.2102,0.2067,0.2627,0.2665


### Event-block bootstrap

The table reports the event-level bootstrap difference `M4 - M3`. Positive intervals in the original and conservative mechanical screens support added value from prior extreme-event response history; intervals crossing zero in the other screens show that this incremental effect depends on event construction.

In [6]:
bootstrap[['scenario','metric','mean_diff','ci_low','ci_high']].round(5)

,scenario,metric,mean_diff,ci_low,ci_high
0,original,PR,0.01816,0.00515,0.03036
1,original,Brier,-0.00038,-0.00061,-0.00019
2,original,AUC,0.00816,-0.00037,0.01824
3,official_action_screened,PR,0.00048,-0.01502,0.01423
4,official_action_screened,Brier,-0.00028,-0.00051,-0.00006
5,official_action_screened,AUC,0.00442,-0.00303,0.01442
6,proxy_consensus_3of4,PR,0.00418,-0.00414,0.01249
7,proxy_consensus_3of4,Brier,-0.00024,-0.00047,-0.00004
8,proxy_consensus_3of4,AUC,0.00525,-0.00284,0.01454
9,mechanical_conservative,PR,0.01625,0.00355,0.02901


## Takeaways

- **Stable finding:** a broad profile label alone does not improve out-of-time ranking consistently.
- **Stable finding:** general revealed transaction behavior is more useful than the stated profile across all event screens.
- **Qualified finding:** prior extreme-event response history can add predictive value, but its incremental magnitude is sensitive to which events are retained.
- **Submission wording:** describe the outcome as an extreme negative-return event, not an exogenous causal shock.
- **Remaining access limitation:** an exact official-index rerun requires the Bank of Greece/ATHEX daily workbook to be supplied from an environment that can download the binary file.